# CS781 Capstone -- Phase 4: Court Judgment Prediction (CJPE)

Binary classification (0/1) over full legal case documents. Four experiments in this first
batch, one per Phase 1 model, each with a technique well suited to it. KD and a 5th technique
come later to complete the required minimum of 6.

| # | Model | Technique |
|---|---|---|
| 1 | GQA | LoRMA |
| 2 | GPT-2 | Full fine-tuning |
| 3 | SwiGLU | BitFit |
| 4 | Prefix-LM | Adapters (hand-rolled Houlsby-style bottleneck) |


## 1. Hugging Face Login

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF2")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 1
GPU 0: Tesla T4


## 3. Configuration

In [ ]:
import os, random, json, time, math, zipfile, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILE = "Phase_4_CJPE/Train/train.json"
DEV_FILE   = "Phase_4_CJPE/Train/dev.json"
TEST_FILE  = "Phase_4_CJPE/Test/test.json"

MODEL_PATHS = {
    "gpt2":    "/kaggle/input/datasets/himangshumahato/model1",
    "swiglu":  "/kaggle/input/datasets/himangshumahato/model2",
    "gqa":     "/kaggle/input/datasets/himangshumahato/model3",
    "prefixlm":"/kaggle/input/datasets/himangshumahato/model4",
}
MODEL_CONTEXT_LENGTH = 1024
N_EMBD, N_LAYER, N_HEAD = 768, 12, 12
N_KV_HEAD = 4

HEAD_TOKENS, TAIL_TOKENS = 256, 256
DOC_MAX_LEN = HEAD_TOKENS + TAIL_TOKENS   # == MODEL_CONTEXT_LENGTH

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_OUTPUT_DIR = "/kaggle/working/cs781_phase4"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

def experiment_dirs(name):
    d = os.path.join(BASE_OUTPUT_DIR, name)
    os.makedirs(d, exist_ok=True)
    return {
        "dir": d,
        "model_path": os.path.join(d, "best_model.pt"),
        "run_info_path": os.path.join(d, "run_info.json"),
        "prediction_path": os.path.join(d, "prediction.json"),
        "submission_zip": os.path.join(d, "submission.zip"),
    }

print("Configuration loaded.")

Configuration loaded.


## 4. Install peft (kept available for a later AdaLoRA experiment; not used by these 4)

In [ ]:
!pip uninstall -y torchao -q 2>/dev/null
!pip -q install regex peft accelerate
import peft
print("peft version:", peft.__version__)

peft version: 0.20.0


## 5. Tokenizer (same shared tokenizer as Phases 1-3)

In [ ]:
%%writefile a1_tokenizer.py
"""From-scratch byte-level BPE tokenizer (GPT-2 style), no external tokenizer libs.

Differences from the plain-bytes version:
  * Text is first split into "words" with a GPT-2 style regex, and merges never
    cross word boundaries. Training and encoding then work on *unique words*
    (with counts) instead of the whole corpus, which is orders of magnitude faster.
  * Training uses incremental pair counts + a lazy max-heap (no full rescan per merge).
  * Encoding applies merges per word by rank, with a word -> ids cache.
  * One special token, <|endoftext|>, used as EOS / PAD / BOS.

Interface:
    tok.encode(text, add_eos=False) -> list[int]
    tok.decode(ids, skip_special_tokens=True) -> str
    tok.vocab_size / tok.get_vocab_size()
    tok.eos_token_id, tok.pad_token_id, tok.bos_token_id
    tok.save(path) / ByteLevelBPE.load(path)
"""

from __future__ import annotations

import heapq
import json
import time
from collections import Counter, defaultdict

import regex as re  # `pip install regex` (needed for \p{L} / \p{N})

# GPT-2 pre-tokenization pattern
_PAT = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

EOS_TOKEN = "<|endoftext|>"


class ByteLevelBPE:
    def __init__(self):
        # ids 0-255 = raw bytes; 256.. = learned merges; last id = <|endoftext|>
        self.merges: dict[tuple[int, int], int] = {}      # (a, b) -> new_id (id order == rank order)
        self.itos: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.eos_token_id = 256
        self.vocab_size = 257
        self._cache: dict[str, tuple] = {}

    # alias attributes so the tokenizer plugs into HF-style code
    @property
    def pad_token_id(self): return self.eos_token_id
    @property
    def bos_token_id(self): return self.eos_token_id

    def __len__(self): return self.vocab_size
    def get_vocab_size(self): return self.vocab_size

    def train(self, text: str, vocab_size: int, verbose: bool = False) -> None:
        """Learn (vocab_size - 257) merges; the final id is reserved for <|endoftext|>."""
        if vocab_size < 257:
            raise ValueError("vocab_size must be at least 257 (256 bytes + EOS)")
        num_merges = vocab_size - 257
        t0 = time.time()

        # 1) pre-tokenize and count unique words
        word_counts = Counter(_PAT.findall(text))
        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        if verbose:
            print(f"[bpe] {len(words):,} unique words from {sum(freqs):,} total words")

        # 2) initial pair statistics
        stats: dict[tuple[int, int], int] = defaultdict(int)
        where: dict[tuple[int, int], set] = defaultdict(set)
        for wi, w in enumerate(words):
            f = freqs[wi]
            for p in zip(w, w[1:]):
                stats[p] += f
                where[p].add(wi)

        # lazy max-heap: (-count, a, b) -> highest count first, ties -> smallest (a, b)
        heap = [(-c, a, b) for (a, b), c in stats.items()]
        heapq.heapify(heap)

        self.merges, self.itos = {}, {i: bytes([i]) for i in range(256)}

        # 3) merge loop
        for i in range(num_merges):
            pair, count = None, 0
            while heap:
                negc, a, b = heapq.heappop(heap)
                if stats.get((a, b), 0) == -negc:   # skip stale heap entries
                    pair, count = (a, b), -negc
                    break
            if pair is None or count < 2:
                break

            new_id = 256 + i
            self.merges[pair] = new_id
            self.itos[new_id] = self.itos[pair[0]] + self.itos[pair[1]]

            touched = set()
            for wi in list(where[pair]):
                w, f = words[wi], freqs[wi]
                # remove this word's old pair contributions
                for p in zip(w, w[1:]):
                    stats[p] -= f
                    where[p].discard(wi)
                    touched.add(p)
                # apply merge (leftmost, non-overlapping)
                a, b = pair
                out, j, n = [], 0, len(w)
                while j < n:
                    if j < n - 1 and w[j] == a and w[j + 1] == b:
                        out.append(new_id); j += 2
                    else:
                        out.append(w[j]); j += 1
                words[wi] = out
                # add new pair contributions
                for p in zip(out, out[1:]):
                    stats[p] += f
                    where[p].add(wi)
                    touched.add(p)

            for p in touched:
                c = stats.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p[0], p[1]))
                else:
                    stats.pop(p, None)
                    where.pop(p, None)

            if verbose and (i % 1000 == 0 or i == num_merges - 1):
                print(f"[bpe] merge {i + 1}/{num_merges} | pair={pair} count={count:,} "
                      f"| elapsed={time.time() - t0:.1f}s")

        self.eos_token_id = 256 + len(self.merges)
        self.vocab_size = self.eos_token_id + 1
        self._cache = {}

    # Inference
    def _encode_word(self, word: str) -> tuple:
        hit = self._cache.get(word)
        if hit is not None:
            return hit
        ids = list(word.encode("utf-8"))
        merges = self.merges
        while len(ids) > 1:
            # pick the applicable merge learned earliest (lowest id)
            best, best_id = None, None
            for p in zip(ids, ids[1:]):
                nid = merges.get(p)
                if nid is not None and (best_id is None or nid < best_id):
                    best, best_id = p, nid
            if best is None:
                break
            a, b = best
            out, j, n = [], 0, len(ids)
            while j < n:
                if j < n - 1 and ids[j] == a and ids[j + 1] == b:
                    out.append(best_id); j += 2
                else:
                    out.append(ids[j]); j += 1
            ids = out
        res = tuple(ids)
        if len(self._cache) > 2_000_000:
            self._cache.clear()
        self._cache[word] = res
        return res

    def encode(self, text: str, add_eos: bool = False) -> list[int]:
        ids: list[int] = []
        for w in _PAT.findall(text):
            ids.extend(self._encode_word(w))
        if add_eos:
            ids.append(self.eos_token_id)
        return ids

    def decode(self, ids, skip_special_tokens: bool = True) -> str:
        if hasattr(ids, "tolist"):
            ids = ids.tolist()
        parts = []
        for i in ids:
            if i == self.eos_token_id:
                if not skip_special_tokens:
                    parts.append(EOS_TOKEN.encode("utf-8"))
                continue
            parts.append(self.itos[i])
        return b"".join(parts).decode("utf-8", errors="replace")

    def save(self, path: str) -> None:
        data = {
            "merges": [[a, b, nid] for (a, b), nid in self.merges.items()],
            "vocab_size": self.vocab_size,
            "eos_token_id": self.eos_token_id,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    @classmethod
    def load(cls, path: str) -> "ByteLevelBPE":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        for a, b, nid in data["merges"]:
            tok.merges[(a, b)] = nid
            tok.itos[nid] = tok.itos[a] + tok.itos[b]
        tok.eos_token_id = data["eos_token_id"]
        tok.vocab_size = data["vocab_size"]
        return tok


def train_tokenizer(train_text: str, vocab_size: int):
    """Train a byte-level BPE tokenizer on `train_text` and return it."""
    tok = ByteLevelBPE()
    tok.train(train_text, vocab_size, verbose=True)
    return tok

Overwriting a1_tokenizer.py


In [ ]:
from a1_tokenizer import ByteLevelBPE, _PAT

tokenizer = ByteLevelBPE.load(os.path.join(MODEL_PATHS["gqa"], "bpe_tokenizer.json"))
print("Vocabulary size:", len(tokenizer))
for name, path in MODEL_PATHS.items():
    t = ByteLevelBPE.load(os.path.join(path, "bpe_tokenizer.json"))
    assert t.vocab_size == tokenizer.vocab_size, f"{name} tokenizer differs!"
print("All four checkpoints share the same tokenizer.")

Vocabulary size: 32000
All four checkpoints share the same tokenizer.


## 6. Load Phase 4 data

In [ ]:
from huggingface_hub import hf_hub_download

def hf_json(path):
    p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)
    with open(p, encoding="utf-8") as f:
        return json.load(f)

train_raw = hf_json(TRAIN_FILE)
dev_raw   = hf_json(DEV_FILE)
test_raw  = hf_json(TEST_FILE)

print("train:", len(train_raw), type(train_raw))
print("dev:", len(dev_raw), type(dev_raw))
print("test:", len(test_raw), type(test_raw))

# inspect actual shape/keys before assuming a label field name
sample = train_raw[0] if isinstance(train_raw, list) else next(iter(train_raw.values()))
print("sample keys:", sample if isinstance(sample, dict) and len(str(sample)) < 300 else {k: str(v)[:80] for k,v in sample.items()})

train: 37387 <class 'list'>
dev: 3505 <class 'list'>
test: 1517 <class 'list'>
sample keys: {'id': '2020_1', 'text': ' Uday Umesh Lalit, J. These appeals arise out of the Judgment and Order dated 09', 'label': '0', 'expert_1': 'None', 'expert_2': 'None', 'expert_3': 'None', 'expert_4': 'None', 'expert_5': 'None'}


## 7. Normalize into records

Handles the label field under any of the plausible key names the dataset might use, and both
list-of-dicts and dict-of-dicts document shapes (as seen in Phase 3's test files) -- checked
explicitly rather than assumed, given past surprises with this dataset's JSON shapes.

In [ ]:
LABEL_KEYS = ["label", "pred", "judgment", "outcome", "y"]

def find_label_key(example):
    for k in LABEL_KEYS:
        if k in example:
            return k
    return None

def to_records(raw, with_labels=True):
    items = raw.items() if isinstance(raw, dict) else [(d.get("id", d.get("case_id")), d) for d in raw]
    records = []
    label_key = None
    for doc_id, d in items:
        if isinstance(d, str):
            text = d
            label = None
        else:
            text = d.get("text", d.get("case_text", d.get("document")))
            if with_labels and label_key is None:
                label_key = find_label_key(d)
                assert label_key is not None, f"no recognized label key in {list(d.keys())}"
            label = d.get(label_key) if with_labels else None
        rec = {"id": doc_id, "text": text}
        if with_labels:
            rec["label"] = int(label)
        records.append(rec)
    return records

train_records = to_records(train_raw, with_labels=True)
dev_records   = to_records(dev_raw,   with_labels=True)
test_records  = to_records(test_raw,  with_labels=False)

print("train:", len(train_records), "| dev:", len(dev_records), "| test:", len(test_records))
print("train label distribution:", {l: sum(1 for r in train_records if r["label"] == l) for l in (0, 1)})
print("dev label distribution:", {l: sum(1 for r in dev_records if r["label"] == l) for l in (0, 1)})

train: 37387 | dev: 3505 | test: 1517
train label distribution: {0: 22067, 1: 15320}
dev label distribution: {0: 1749, 1: 1756}


## 8. Document length distribution (justifies the head+tail choice)

In [ ]:
lengths = np.array([len(tokenizer.encode(r["text"])) for r in train_records[:200]])   # sample for speed
for p in [50, 75, 90, 95, 99, 100]:
    print(f"p{p:>4}: {np.percentile(lengths, p):.0f} tokens")
over_limit = (lengths > DOC_MAX_LEN).mean()
print(f"fraction of (sampled) documents exceeding {DOC_MAX_LEN} tokens: {over_limit:.1%}")

p  50: 4127 tokens
p  75: 6384 tokens
p  90: 11300 tokens
p  95: 20571 tokens
p  99: 59122 tokens
p 100: 125397 tokens
fraction of (sampled) documents exceeding 512 tokens: 98.5%


## 9. Dataset / collator -- head+tail truncation

In [ ]:
PAD_ID = tokenizer.eos_token_id

def encode_head_tail(text, head=HEAD_TOKENS, tail=TAIL_TOKENS):
    ids = tokenizer.encode(text)
    if len(ids) <= head + tail:
        return ids
    return ids[:head] + ids[-tail:]


class CJPEDataset(torch.utils.data.Dataset):
    def __init__(self, records, with_labels=True):
        self.records = records
        self.with_labels = with_labels

    def __len__(self):
        return len(self.records)

    # def __getitem__(self, idx):
    #     r = self.records[idx]
    #     ids = encode_head_tail(r["text"])
    #     item = {"ids": ids, "idx": idx}
    #     if self.with_labels:
    #         item["label"] = r["label"]
    #     return item
    def __getitem__(self, idx):
       r = self.records[idx]
       if "ids" not in r:                                   # tokenize once, keep for later epochs
           r["ids"] = np.array(encode_head_tail(r["text"]), dtype=np.int32)
       item = {"ids": r["ids"], "idx": idx}
       if self.with_labels:
           item["label"] = r["label"]
       return item


def cjpe_collator(batch):
    L = DOC_MAX_LEN
    B = len(batch)
    input_ids = torch.full((B, L), PAD_ID, dtype=torch.long)
    attention_mask = torch.zeros((B, L), dtype=torch.bool)
    for i, item in enumerate(batch):
        n = len(item["ids"])
        input_ids[i, :n] = torch.tensor(item["ids"])
        attention_mask[i, :n] = True
    out = {"input_ids": input_ids, "attention_mask": attention_mask,
           "record_idx": torch.tensor([item["idx"] for item in batch])}
    if "label" in batch[0]:
        out["labels"] = torch.tensor([item["label"] for item in batch])
    return out

_t = cjpe_collator([CJPEDataset(train_records)[i] for i in range(4)])
print({k: v.shape for k, v in _t.items()})

{'input_ids': torch.Size([4, 512]), 'attention_mask': torch.Size([4, 512]), 'record_idx': torch.Size([4]), 'labels': torch.Size([4])}


## 10. Class weights

In [ ]:
from collections import Counter

train_label_counts = Counter(r["label"] for r in train_records)
total = sum(train_label_counts.values())
print("label counts:", dict(train_label_counts))
weights_list = [total / (2 * train_label_counts.get(l, 1)) for l in (0, 1)]
class_weights = torch.tensor(weights_list, dtype=torch.float).to(device)
print("class weights:", class_weights.tolist())

label counts: {0: 22067, 1: 15320}
class weights: [0.8471246361732483, 1.2202023267745972]


## 11. Backbone architectures (same four classes as Phase 2/3)

In [ ]:
import math as _m

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.n_head = n_head; self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout); self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(n_positions, n_positions)).view(1, 1, n_positions, n_positions))

    def forward(self, x, layer_past=None, attn_mask=None):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if layer_past is not None:
            pk, pv = layer_past; k = torch.cat([pk, k], -2); v = torch.cat([pv, v], -2)
        present = (k, v)
        past_len = layer_past[0].size(-2) if layer_past is not None else 0
        seq_len = T + past_len
        att = (q @ k.transpose(-2, -1)) * (1.0 / _m.sqrt(self.head_dim))
        if attn_mask is not None:
            att = att.masked_fill(~attn_mask, float('-inf'))
        else:
            mask = self.bias[:, :, past_len:past_len + T, :seq_len] == 0
            att = att.masked_fill(mask, float('-inf'))
        att = self.attn_dropout(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.c_fc = nn.Linear(n_embd, 4 * n_embd); self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * n_embd, n_embd); self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd); self.attn = CausalSelfAttention(n_embd, n_head, n_positions, dropout)
        self.ln_2 = nn.LayerNorm(n_embd); self.mlp = MLP(n_embd, dropout)

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class GPT2ScratchLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([TransformerBlock(n_embd, n_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


class SwiGLUMLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1, hidden_mult=4):
        super().__init__()
        hidden = int(n_embd * hidden_mult * 2 / 3)
        self.w1 = nn.Linear(n_embd, hidden, bias=False)
        self.w3 = nn.Linear(n_embd, hidden, bias=False)
        self.w2 = nn.Linear(hidden, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class CausalSelfAttentionRMS(CausalSelfAttention):
    pass


class RMSSwiGLUBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = RMSNorm(n_embd); self.attn = CausalSelfAttentionRMS(n_embd, n_head, n_positions, dropout)
        self.ln_2 = RMSNorm(n_embd); self.mlp = SwiGLUMLP(n_embd, dropout)

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class RMSSwiGLULM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([RMSSwiGLUBlock(n_embd, n_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


class GroupedQueryAttention(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, n_positions, dropout=0.1):
        super().__init__()
        self.n_head = n_head; self.n_kv_head = n_kv_head
        self.head_dim = n_embd // n_head; self.num_groups = n_head // n_kv_head
        self.q_proj = nn.Linear(n_embd, n_embd)
        self.k_proj = nn.Linear(n_embd, n_kv_head * self.head_dim)
        self.v_proj = nn.Linear(n_embd, n_kv_head * self.head_dim)
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout); self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("bias", torch.tril(torch.ones(n_positions, n_positions)).view(1, 1, n_positions, n_positions))

    def forward(self, x, layer_past=None, attn_mask=None):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        if layer_past is not None:
            pk, pv = layer_past; k = torch.cat([pk, k], -2); v = torch.cat([pv, v], -2)
        present = (k, v)
        past_len = layer_past[0].size(-2) if layer_past is not None else 0
        seq_len = T + past_len
        k = k.repeat_interleave(self.num_groups, dim=1); v = v.repeat_interleave(self.num_groups, dim=1)
        att = (q @ k.transpose(-2, -1)) * (1.0 / _m.sqrt(self.head_dim))
        if attn_mask is not None:
            att = att.masked_fill(~attn_mask, float('-inf'))
        else:
            mask = self.bias[:, :, past_len:past_len + T, :seq_len] == 0
            att = att.masked_fill(mask, float('-inf'))
        att = self.attn_dropout(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y)), present


class GQATransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, n_kv_head, n_positions, dropout=0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(n_embd)
        self.attn = GroupedQueryAttention(n_embd, n_head, n_kv_head, n_positions, dropout)
        self.ln_2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.GELU(approximate='tanh'),
                                 nn.Linear(4*n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x, layer_past=None, attn_mask=None):
        a, present = self.attn(self.ln_1(x), layer_past=layer_past, attn_mask=attn_mask)
        x = x + a; x = x + self.mlp(self.ln_2(x))
        return x, present


class ScratchGroupedQueryLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, n_kv_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([GQATransformerBlock(n_embd, n_head, n_kv_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)


class PrefixLM(nn.Module):
    def __init__(self, vocab_size, n_positions, n_embd, n_layer, n_head, n_kv_head, dropout=0.1):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, n_embd); self.wpe = nn.Embedding(n_positions, n_embd)
        self.drop = nn.Dropout(dropout)
        self.h = nn.ModuleList([GQATransformerBlock(n_embd, n_head, n_kv_head, n_positions, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd); self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

print("Backbone classes defined.")

Backbone classes defined.


## 12. Backbone registry and loader

In [ ]:
from safetensors.torch import load_file

BACKBONE_REGISTRY = {
    "gpt2":     {"cls": GPT2ScratchLM,       "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD)},
    "swiglu":   {"cls": RMSSwiGLULM,         "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD)},
    "gqa":      {"cls": ScratchGroupedQueryLM, "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD)},
    "prefixlm": {"cls": PrefixLM,            "kwargs": dict(n_embd=N_EMBD, n_layer=N_LAYER, n_head=N_HEAD, n_kv_head=N_KV_HEAD)},
}

def load_backbone(name):
    spec = BACKBONE_REGISTRY[name]
    model = spec["cls"](vocab_size=tokenizer.vocab_size, n_positions=MODEL_CONTEXT_LENGTH, **spec["kwargs"])
    state = load_file(os.path.join(MODEL_PATHS[name], "model.safetensors"))
    result = model.load_state_dict(state)
    print(f"[{name}] load_state_dict: {result}")
    if hasattr(model, "lm_head"):
        del model.lm_head
    return model

for name in BACKBONE_REGISTRY:
    m = load_backbone(name)
    print(f"{name}: {sum(p.numel() for p in m.parameters())/1e6:.1f}M params")
    del m

[gpt2] load_state_dict: <All keys matched successfully>
gpt2: 110.4M params
[swiglu] load_state_dict: <All keys matched successfully>
swiglu: 110.4M params
[gqa] load_state_dict: <All keys matched successfully>
gqa: 101.0M params
[prefixlm] load_state_dict: <All keys matched successfully>
prefixlm: 101.0M params


## 13. Document classifier wrapper (last+mean pooling -> 2-class head)

In [ ]:
class DocClassifier(nn.Module):
    def __init__(self, lm, n_embd, dropout=0.1):
        super().__init__()
        self.lm = lm
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(2 * n_embd, n_embd), nn.GELU(approximate='tanh'),
            nn.Dropout(dropout), nn.Linear(n_embd, 2)
        )

    def forward(self, input_ids, attention_mask=None, labels=None, class_weights=None, threshold_offset=None):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device)
        x = self.lm.wte(input_ids) + self.lm.wpe(positions)
        x = self.lm.drop(x)

        causal = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device)).view(1, 1, T, T)
        attn_mask = causal
        if attention_mask is not None:
            attn_mask = causal & attention_mask[:, None, None, :].bool()

        for block in self.lm.h:
            x, _ = block(x, attn_mask=attn_mask)
        x = self.lm.ln_f(x)

        if attention_mask is not None:
            last_idx = attention_mask.sum(dim=1) - 1
            m = attention_mask.unsqueeze(-1).float()
        else:
            last_idx = torch.full((B,), T - 1, device=input_ids.device)
            m = torch.ones(B, T, 1, device=input_ids.device)
        last = x[torch.arange(B, device=input_ids.device), last_idx]
        mean = ((x.float() * m).sum(dim=1) / m.sum(dim=1).clamp(min=1)).to(x.dtype)
        pooled = torch.cat([last, mean], dim=-1)

        logits = self.head(self.dropout(pooled))
        if threshold_offset is not None:
            logits = logits + threshold_offset   # shifts class-1 log-odds, equivalent to a decision threshold
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels, weight=class_weights)
        return {"loss": loss, "logits": logits}

print("DocClassifier defined.")

DocClassifier defined.


## 14. Metrics and threshold calibration

Binary macro-F1/P/R, plus a false-positive/false-negative breakdown (the brief's Section 10.5
explicitly asks for this). Threshold tuning follows the same half/half dev-generalization check
used in Phase 3's logit-offset calibration, so it isn't overfit to the small dev set.

In [ ]:
def macro_prf_binary(y_true, y_pred):
    cnts = np.zeros((2, 3))
    for t, p in zip(y_true, y_pred):
        cnts[p, 1] += 1; cnts[t, 2] += 1
        if t == p: cnts[t, 0] += 1
    P = np.divide(cnts[:, 0], cnts[:, 1], out=np.zeros(2), where=cnts[:, 1] > 0)
    R = np.divide(cnts[:, 0], cnts[:, 2], out=np.zeros(2), where=cnts[:, 2] > 0)
    F = np.divide(2 * P * R, P + R, out=np.zeros(2), where=(P + R) > 0)
    return P, R, F


def print_binary_scores(y_true, y_pred, title=""):
    print(f"--- {title} ---")
    P, R, F = macro_prf_binary(y_true, y_pred)
    for i in (0, 1):
        print(f"class {i}: P {P[i]:.3f}  R {R[i]:.3f}  F1 {F[i]:.3f}")
    print(f"MACRO  P {P.mean():.4f}  R {R.mean():.4f}  F1 {F.mean():.4f}")
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
    tn = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
    print(f"confusion: TP={tp} FP={fp} FN={fn} TN={tn}")
    return F.mean()


def raw_macro_f1_binary(logits, labels):
    preds = logits.argmax(-1).numpy()
    return macro_prf_binary(labels.numpy(), preds)[2].mean()


@torch.no_grad()
def get_logits(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"])
        all_logits.append(out["logits"].float().cpu())
        if "labels" in b:
            all_labels.append(b["labels"].cpu())
    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels) if all_labels else None
    return logits, labels


def tune_threshold_offset(logits, labels, n_steps=41):
    y = labels.numpy()
    offsets = np.linspace(-3.0, 3.0, n_steps)
    best_off, best_f1 = 0.0, raw_macro_f1_binary(logits, labels)
    for off in offsets:
        shifted = logits.clone(); shifted[:, 1] += off
        f1 = macro_prf_binary(y, shifted.argmax(-1).numpy())[2].mean()
        if f1 > best_f1:
            best_off, best_f1 = off, f1
    return best_off, best_f1


def threshold_calibration_check(logits, labels, min_gain=0.005):
    n = len(labels)
    idx = torch.randperm(n, generator=torch.Generator().manual_seed(SEED))
    half = n // 2
    a, b = idx[:half], idx[half:]
    off_a, _ = tune_threshold_offset(logits[a], labels[a])
    shifted_b = logits[b].clone(); shifted_b[:, 1] += off_a
    f1_b_with = macro_prf_binary(labels[b].numpy(), shifted_b.argmax(-1).numpy())[2].mean()
    f1_b_without = raw_macro_f1_binary(logits[b], labels[b])

    off_b, _ = tune_threshold_offset(logits[b], labels[b])
    shifted_a = logits[a].clone(); shifted_a[:, 1] += off_b
    f1_a_with = macro_prf_binary(labels[a].numpy(), shifted_a.argmax(-1).numpy())[2].mean()
    f1_a_without = raw_macro_f1_binary(logits[a], labels[a])

    gain = ((f1_b_with - f1_b_without) + (f1_a_with - f1_a_without)) / 2
    print(f"held-out half: F1 without offset {f1_b_without:.4f} | with offset {f1_b_with:.4f}")
    print(f"held-out half: F1 without offset {f1_a_without:.4f} | with offset {f1_a_with:.4f}")
    print(f"mean held-out gain from threshold offset: {gain:+.4f}")
    return gain

print("Metrics and calibration utilities defined.")

Metrics and calibration utilities defined.


## 15. Submission writer

Phase 4's format is a JSON list of `{"id", "pred"}` objects -- different from Phases 1-3.

In [ ]:
def write_submission(paths, model, threshold_offset=None):
    test_ds = CJPEDataset(test_records, with_labels=False)
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=8, shuffle=False, collate_fn=cjpe_collator)

    model.eval()
    all_logits = []
    with torch.no_grad():
        for b in test_loader:
            b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(input_ids=b["input_ids"], attention_mask=b["attention_mask"])
            all_logits.append(out["logits"].float().cpu())
    logits = torch.cat(all_logits)
    if threshold_offset:
        logits[:, 1] += threshold_offset
    preds = logits.argmax(-1).tolist()

    prediction = [{"id": r["id"], "pred": int(p)} for r, p in zip(test_records, preds)]
    assert len(prediction) == len(test_records)
    assert all(p["pred"] in (0, 1) for p in prediction)
    assert {p["id"] for p in prediction} == {r["id"] for r in test_records}

    with open(paths["prediction_path"], "w", encoding="utf-8") as f:
        json.dump(prediction, f, ensure_ascii=False)
    with zipfile.ZipFile(paths["submission_zip"], "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(paths["prediction_path"], arcname="prediction.json")
    print("saved:", paths["submission_zip"])
    with zipfile.ZipFile(paths["submission_zip"]) as z:
        print(z.namelist())
    return prediction

print("Submission writer defined.")

Submission writer defined.


## 16. Generic training loop

In [ ]:
def train_classifier(model, optimizer, scheduler, train_loader, dev_loader,
                     class_weights, epochs, patience, paths, grad_accum=1, min_cal_gain=0.005):
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    history = {"train": [], "dev_f1": []}
    best_f1, bad_epochs = 0.0, 0
    t0 = time.time()

    for epoch in range(epochs):
        model.train()
        run, cnt = 0.0, 0
        optimizer.zero_grad()
        for i, b in enumerate(train_loader):
            b = {k: v.to(device) for k, v in b.items() if k != "record_idx"}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(**b, class_weights=class_weights)
            scaler.scale(out["loss"] / grad_accum).backward()
            run += out["loss"].item(); cnt += 1
            if (i + 1) % grad_accum == 0 or (i + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad()
                if scheduler is not None:
                    scheduler.step()

        logits, labels = get_logits(model, dev_loader)
        f1 = raw_macro_f1_binary(logits, labels)
        history["train"].append(run / cnt); history["dev_f1"].append(f1)
        print(f"epoch {epoch+1}/{epochs} | train loss {run/cnt:.4f} | dev macro-F1 (uncalibrated) {f1:.4f} | {time.time()-t0:.0f}s")

        if f1 > best_f1:
            best_f1, bad_epochs = f1, 0
            torch.save(model.state_dict(), paths["model_path"])
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"early stopping at epoch {epoch+1} (best uncalibrated dev macro-F1 {best_f1:.4f})")
                break

    model.load_state_dict(torch.load(paths["model_path"], weights_only=True))
    logits, labels = get_logits(model, dev_loader)

    print("threshold calibration check:")
    gain = threshold_calibration_check(logits, labels, min_cal_gain)
    if gain > min_cal_gain:
        offset, _ = tune_threshold_offset(logits, labels)
        title = "FINAL (best checkpoint, threshold calibrated)"
    else:
        offset = 0.0
        title = "FINAL (best checkpoint, no threshold adjustment)"
        print("threshold offset not used")

    shifted = logits.clone(); shifted[:, 1] += offset
    preds = shifted.argmax(-1).tolist()
    final_f1 = print_binary_scores(labels.tolist(), preds, title=title)
    print("best uncalibrated dev macro-F1:", best_f1, "| final reported:", float(final_f1))

    return model, history, offset, float(best_f1), float(final_f1)

print("train_classifier defined.")

train_classifier defined.


---
## Experiment 1 -- GQA + LoRMA

Same from-scratch LoRMA implementation as Phases 2-3: `W' = (I + (alpha/r) B@A) @ W0`. Targets
`q_proj`/`v_proj`, backbone frozen, only LoRMA matrices + head trainable.

In [ ]:
class LoRMALinear(nn.Module):
    def __init__(self, base_linear, r=8, alpha=16):
        super().__init__()
        self.base = base_linear
        for p in self.base.parameters():
            p.requires_grad = False
        d_out = base_linear.out_features
        self.r = r; self.alpha = alpha
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.A = nn.Parameter(torch.randn(r, d_out) * 0.01)
        self.register_buffer("eye", torch.eye(d_out))

    def forward(self, x):
        BA = self.B @ self.A
        M = self.eye + (self.alpha / self.r) * BA
        W_eff = M @ self.base.weight
        return F.linear(x, W_eff, self.base.bias)


def apply_lorma(model, target_names, r=8, alpha=16):
    for parent_name, parent in model.named_modules():
        for child_name, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and child_name in target_names:
                setattr(parent, child_name, LoRMALinear(child, r=r, alpha=alpha))
    return model

print("LoRMA defined.")

LoRMA defined.


In [ ]:
EXP1_NAME = "01_gqa_lorma"
paths = experiment_dirs(EXP1_NAME)

LORMA_R, LORMA_ALPHA = 8, 16
LORMA_TARGETS = ["q_proj", "v_proj"]
EPOCHS, BATCH_SIZE, GRAD_ACCUM = 10, 8, 2
LR_LORMA, LR_HEAD, WEIGHT_DECAY, WARMUP_RATIO, PATIENCE = 1e-3, 1e-3, 0.05, 0.10, 3

backbone = load_backbone("gqa")
for p in backbone.parameters():
    p.requires_grad = False
backbone = apply_lorma(backbone, LORMA_TARGETS, r=LORMA_R, alpha=LORMA_ALPHA)
model = DocClassifier(backbone, n_embd=N_EMBD).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_loader = torch.utils.data.DataLoader(CJPEDataset(train_records), batch_size=BATCH_SIZE,
                                           shuffle=True, collate_fn=cjpe_collator)
dev_loader = torch.utils.data.DataLoader(CJPEDataset(dev_records), batch_size=8,
                                         shuffle=False, collate_fn=cjpe_collator)

lorma_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" in n]
optimizer = torch.optim.AdamW([
    {"params": lorma_params, "lr": LR_LORMA},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps)

[gqa] load_state_dict: <All keys matched successfully>
trainable: 1,378,562 / 102,347,522 (1.35%)
optimizer steps: 23370


In [ ]:
model, history, offset, best_f1, final_f1 = train_classifier(
    model, optimizer, scheduler, train_loader, dev_loader,
    class_weights, EPOCHS, PATIENCE, paths, grad_accum=GRAD_ACCUM
)

In [ ]:
run_info = {
    "experiment": EXP1_NAME, "phase1_model": "GQA", "technique": "LoRMA",
    "r": LORMA_R, "alpha": LORMA_ALPHA, "targets": LORMA_TARGETS,
    "head_tokens": HEAD_TOKENS, "tail_tokens": TAIL_TOKENS,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "total_params": total, "threshold_offset": offset,
    "pooling": "last+mean", "selection_metric": "uncalibrated dev macro-F1",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
prediction = write_submission(paths, model, offset)

---
## Experiment 2 -- GPT-2 + Full Fine-tuning

Every backbone parameter trainable, low LR (standard full-FT discipline from Phase 2) + higher
LR for the new head.

In [ ]:
EXP2_NAME = "02_gpt2_fullfinetune"
paths = experiment_dirs(EXP2_NAME)

EPOCHS, BATCH_SIZE, GRAD_ACCUM = 15, 8, 2

LR_BACKBONE, LR_HEAD, WEIGHT_DECAY, WARMUP_RATIO, PATIENCE = 3e-5, 1e-3, 0.05, 0.10, 5

backbone = load_backbone("gpt2")
for p in backbone.parameters():
    p.requires_grad = True
model = DocClassifier(backbone, n_embd=N_EMBD).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_loader = torch.utils.data.DataLoader(CJPEDataset(train_records), batch_size=BATCH_SIZE,
                                           shuffle=True, collate_fn=cjpe_collator)
dev_loader = torch.utils.data.DataLoader(CJPEDataset(dev_records), batch_size=8,
                                         shuffle=False, collate_fn=cjpe_collator)

backbone_params = [p for n, p in model.named_parameters() if "head" not in n]
head_params = [p for n, p in model.named_parameters() if "head" in n]
optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": LR_BACKBONE},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps)

In [ ]:
model, history, offset, best_f1, final_f1 = train_classifier(
    model, optimizer, scheduler, train_loader, dev_loader,
    class_weights, EPOCHS, PATIENCE, paths, grad_accum=GRAD_ACCUM
)

In [ ]:
run_info = {
    "experiment": EXP2_NAME, "phase1_model": "GPT-2", "technique": "Full Fine-tuning",
    "lr_backbone": LR_BACKBONE, "lr_head": LR_HEAD,
    "head_tokens": HEAD_TOKENS, "tail_tokens": TAIL_TOKENS,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "total_params": total, "threshold_offset": offset,
    "pooling": "last+mean", "selection_metric": "uncalibrated dev macro-F1",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
prediction = write_submission(paths, model, offset)

---
## Experiment 3 -- SwiGLU + BitFit

Only bias terms and normalization gains are trainable (the canonical BitFit definition), plus
the new head -- everything else (all weight matrices) frozen.

In [ ]:
def apply_bitfit(model):
    for n, p in model.named_parameters():
        p.requires_grad = False
    for n, p in model.named_parameters():
        if n.endswith(".bias") or n.endswith("ln_1.weight") or n.endswith("ln_2.weight") or n.endswith("ln_f.weight"):
            p.requires_grad = True
    return model

print("BitFit defined.")

In [ ]:
EXP3_NAME = "03_swiglu_bitfit"
paths = experiment_dirs(EXP3_NAME)

EPOCHS, BATCH_SIZE, GRAD_ACCUM = 15, 8, 2
LR_BITFIT, LR_HEAD, WEIGHT_DECAY, WARMUP_RATIO, PATIENCE = 5e-4, 1e-3, 0.05, 0.10, 5

backbone = load_backbone("swiglu")
backbone = apply_bitfit(backbone)
model = DocClassifier(backbone, n_embd=N_EMBD).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_loader = torch.utils.data.DataLoader(CJPEDataset(train_records), batch_size=BATCH_SIZE,
                                           shuffle=True, collate_fn=cjpe_collator)
dev_loader = torch.utils.data.DataLoader(CJPEDataset(dev_records), batch_size=8,
                                         shuffle=False, collate_fn=cjpe_collator)

bitfit_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" in n]
optimizer = torch.optim.AdamW([
    {"params": bitfit_params, "lr": LR_BITFIT},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps)

In [ ]:
model, history, offset, best_f1, final_f1 = train_classifier(
    model, optimizer, scheduler, train_loader, dev_loader,
    class_weights, EPOCHS, PATIENCE, paths, grad_accum=GRAD_ACCUM
)

In [ ]:
run_info = {
    "experiment": EXP3_NAME, "phase1_model": "SwiGLU", "technique": "BitFit",
    "lr_bitfit": LR_BITFIT, "lr_head": LR_HEAD,
    "head_tokens": HEAD_TOKENS, "tail_tokens": TAIL_TOKENS,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "total_params": total, "threshold_offset": offset,
    "pooling": "last+mean", "selection_metric": "uncalibrated dev macro-F1",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
prediction = write_submission(paths, model, offset)

---
## Experiment 4 -- Prefix-LM + Adapters

Hand-rolled Houlsby-style bottleneck adapter (down-project -> GELU -> up-project, residual-added,
near-identity at init via zero-initialized up-projection), applied after each transformer block's
output. Backbone frozen; only the adapters + head train.

In [ ]:
class AdapterBottleneck(nn.Module):
    def __init__(self, n_embd, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(n_embd, bottleneck)
        self.act = nn.GELU(approximate='tanh')
        self.up = nn.Linear(bottleneck, n_embd)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)   # near-identity at init, like LoRA/LoRMA's zero-init

    def forward(self, x):
        return x + self.up(self.act(self.down(x)))


class AdapterBlockWrapper(nn.Module):
    def __init__(self, block, n_embd, bottleneck=64):
        super().__init__()
        self.block = block
        for p in self.block.parameters():
            p.requires_grad = False
        self.adapter = AdapterBottleneck(n_embd, bottleneck)

    def forward(self, x, layer_past=None, attn_mask=None):
        x, present = self.block(x, layer_past=layer_past, attn_mask=attn_mask)
        x = self.adapter(x)
        return x, present


def apply_adapters(model, n_embd, bottleneck=64):
    for i in range(len(model.h)):
        model.h[i] = AdapterBlockWrapper(model.h[i], n_embd, bottleneck)
    return model

print("Adapters defined.")

In [ ]:
EXP4_NAME = "04_prefixlm_adapters"
paths = experiment_dirs(EXP4_NAME)

ADAPTER_BOTTLENECK = 64
EPOCHS, BATCH_SIZE, GRAD_ACCUM = 15, 8, 2
LR_ADAPTER, LR_HEAD, WEIGHT_DECAY, WARMUP_RATIO, PATIENCE = 1e-3, 1e-3, 0.05, 0.10, 5

backbone = load_backbone("prefixlm")
for p in backbone.parameters():
    p.requires_grad = False
backbone = apply_adapters(backbone, N_EMBD, bottleneck=ADAPTER_BOTTLENECK)
model = DocClassifier(backbone, n_embd=N_EMBD).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

train_loader = torch.utils.data.DataLoader(CJPEDataset(train_records), batch_size=BATCH_SIZE,
                                           shuffle=True, collate_fn=cjpe_collator)
dev_loader = torch.utils.data.DataLoader(CJPEDataset(dev_records), batch_size=8,
                                         shuffle=False, collate_fn=cjpe_collator)

adapter_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" not in n]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and "head" in n]
optimizer = torch.optim.AdamW([
    {"params": adapter_params, "lr": LR_ADAPTER},
    {"params": head_params, "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
def lr_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
print("optimizer steps:", total_steps)

In [ ]:
model, history, offset, best_f1, final_f1 = train_classifier(
    model, optimizer, scheduler, train_loader, dev_loader,
    class_weights, EPOCHS, PATIENCE, paths, grad_accum=GRAD_ACCUM
)

In [ ]:
run_info = {
    "experiment": EXP4_NAME, "phase1_model": "Prefix-LM", "technique": "Adapters (Houlsby bottleneck)",
    "bottleneck_dim": ADAPTER_BOTTLENECK, "lr_adapter": LR_ADAPTER, "lr_head": LR_HEAD,
    "head_tokens": HEAD_TOKENS, "tail_tokens": TAIL_TOKENS,
    "epochs_run": len(history["dev_f1"]), "best_dev_f1": best_f1, "final_dev_f1": final_f1,
    "trainable_params": trainable, "total_params": total, "threshold_offset": offset,
    "pooling": "last+mean", "selection_metric": "uncalibrated dev macro-F1",
}
with open(paths["run_info_path"], "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)
prediction = write_submission(paths, model, offset)

---
## Summary -- these 4 submissions

Run once all four have finished.

In [ ]:
import glob
print(f"{'experiment':30s} {'final_dev_f1':>13s}")
for d in sorted(glob.glob(os.path.join(BASE_OUTPUT_DIR, "*", "run_info.json"))):
    with open(d) as f:
        ri = json.load(f)
    f1 = ri.get("final_dev_f1", ri.get("best_dev_f1", float("nan")))
    print(f"{ri['experiment']:30s} {f1:13.4f}")